In [1]:
import os, sys
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss


def locate_file(*parts):
    candidates = [
        os.path.join(*parts),  # relative to cwd
        os.path.join("kaggle", "input", *parts),  # /kaggle/input/…
        os.path.join("kaggle", "working", *parts),  # /kaggle/working/…
        os.path.join("..", "input", *parts),  # ../input/…
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Unable to locate {'/'.join(parts)}")


train_path = locate_file("leaf-classification", "train.csv")
test_path = locate_file("leaf-classification", "test.csv")
sample_path = locate_file("leaf-classification", "sample_submission.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

train_ids = train_df["id"].copy()
test_ids = test_df["id"].copy()

X = train_df.drop(["id", "species"], axis=1)
y = train_df["species"]




In [2]:
le = LabelEncoder()
y_enc = le.fit_transform(y)  # integer labels

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)




In [3]:
n_classes = len(np.unique(y_enc))
n_samples = X_scaled.shape[0]
min_test_frac = max(0.1, (n_classes + 1) / n_samples)
test_size = max(min_test_frac, 0.2)

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled,
    y_enc,
    test_size=test_size,
    random_state=42,
    shuffle=True,
    stratify=y_enc,
)




In [4]:
# Slightly larger network and a bit less regularisation to improve fit,
# while keeping the same overall architecture and training loop.
model = MLPClassifier(
    hidden_layer_sizes=(768, 384),  # increased capacity
    activation="relu",
    solver="adam",
    learning_rate="adaptive",
    max_iter=2500,  # allow a few more iterations for convergence
    alpha=1e-5,  # weaker L2 regularisation
    random_state=42,
    verbose=False,
)

model.fit(X_train, y_train)




MLPClassifier(alpha=1e-05, hidden_layer_sizes=(768, 384),
              learning_rate='adaptive', max_iter=2500, random_state=42)

In [5]:
val_proba = model.predict_proba(X_val)
val_logits = np.log(val_proba + 1e-15)  # avoid log(0)


def softmax_with_temp(logits, T):
    """Apply temperature scaling to logits."""
    scaled = logits / T
    exp_scaled = np.exp(scaled - np.max(scaled, axis=1, keepdims=True))
    return exp_scaled / np.sum(exp_scaled, axis=1, keepdims=True)


# Expanded temperature search (including values >5) for finer calibration
temps = np.logspace(-2, 1, 100)  # 0.01 … 10
best_T = 1.0
best_loss = np.inf
for T in temps:
    prob_T = softmax_with_temp(val_logits, T)
    loss_T = log_loss(y_val, prob_T, labels=np.arange(len(le.classes_)))
    if loss_T < best_loss:
        best_loss = loss_T
        best_T = T

val_pred_scaled = softmax_with_temp(val_logits, best_T)
val_loss = log_loss(y_val, val_pred_scaled, labels=np.arange(len(le.classes_)))
print(f"Validation log‑loss after temperature scaling (T={best_T:.4f}): {val_loss:.5f}")




Validation log‑loss after temperature scaling (T=0.7565): 0.03301


In [6]:
X_test = test_df.drop(["id"], axis=1)
X_test_scaled = scaler.transform(X_test)

test_proba = model.predict_proba(X_test_scaled)
test_logits = np.log(test_proba + 1e-15)




In [7]:
y_test_pred = softmax_with_temp(test_logits, best_T)

eps = 1e-15
y_test_pred = np.clip(y_test_pred, eps, 1 - eps)




In [8]:
pred_df = pd.DataFrame(y_test_pred, columns=le.classes_)
pred_df = pred_df[sample_sub.columns[1:]]  # align column order with sample submission
pred_df.insert(0, "id", test_ids.values)

submission_path = "submission_nn_kernel.csv"
pred_df.to_csv(submission_path, index=False)

print(f"Submission written to {submission_path}")

Submission written to submission_nn_kernel.csv
